In [10]:
!pip install "transformers[torch]"

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:

from datasets import load_dataset

dataset = load_dataset('csv', data_files={
    'train': '/content/drive/MyDrive/datasets/samsum-train.csv',
    'validation': '/content/drive/MyDrive/datasets/samsum-validation.csv'
})
print(dataset['train'][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


In [15]:

train_data = dataset['train'].shuffle(seed=42).select(range(4000))
val_data = dataset['validation'].shuffle(seed=42).select(range(500))

In [16]:
import re
def clean_data(text):
    text = re.sub(r"\r\n"," ",text)
    text = re.sub(r"\s+"," ",text)
    text = re.sub(r"<.*?>"," ",text)
    text = text.strip().lower()
    return text


In [17]:
def clean_dataset(row):
    row["dialogue"] = clean_data(row["dialogue"])
    row["summary"] = clean_data(row["summary"])
    return row

train_data = train_data.map(clean_dataset)
val_data = val_data.map(clean_dataset)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [18]:
train_data["dialogue"][0]

'lucy: omg did you see jk this morning? sue: i try to avoid it lol lucy: you should have seen it it was disgusting sue: i cant do it anymore i try to listen to the radio in the mornings.. jk makes you think the whole world is full of idiots lol lucy: you may be right i dont know how some of them can go on there in public for the world to see sue: i would die if i got a call to go on there lol sue: could you imagine ha ha lucy: i would piss myself if i saw you and andy up there sue: over my dead body !'

In [19]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [20]:
tokenizer_model = T5Tokenizer.from_pretrained("t5-small")

def tokenize_row(row):
    inputs = tokenizer_model(
        row["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True,
    )
    targets = tokenizer_model(
        row["summary"],
        padding="max_length",
        max_length=150,
        truncation=True,
    )
    inputs["labels"] = targets["input_ids"]
    return inputs




In [21]:

train_data = train_data.map(tokenize_row, batched=False)
val_data = val_data.map(tokenize_row, batched=False)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [22]:
train_data[0]["input_ids"]

[3,
 11525,
 63,
 10,
 3,
 32,
 51,
 122,
 410,
 25,
 217,
 3,
 354,
 157,
 48,
 1379,
 58,
 2629,
 15,
 10,
 3,
 23,
 653,
 12,
 1792,
 34,
 16497,
 3,
 11525,
 63,
 10,
 25,
 225,
 43,
 894,
 34,
 34,
 47,
 27635,
 53,
 2629,
 15,
 10,
 3,
 23,
 54,
 17,
 103,
 34,
 7595,
 3,
 23,
 653,
 12,
 3011,
 12,
 8,
 2252,
 16,
 8,
 1379,
 7,
 5,
 5,
 3,
 354,
 157,
 656,
 25,
 317,
 8,
 829,
 296,
 19,
 423,
 13,
 25851,
 7,
 16497,
 3,
 11525,
 63,
 10,
 25,
 164,
 36,
 269,
 3,
 23,
 2483,
 214,
 149,
 128,
 13,
 135,
 54,
 281,
 30,
 132,
 16,
 452,
 21,
 8,
 296,
 12,
 217,
 2629,
 15,
 10,
 3,
 23,
 133,
 67,
 3,
 99,
 3,
 23,
 530,
 3,
 9,
 580,
 12,
 281,
 30,
 132,
 16497,
 2629,
 15,
 10,
 228,
 25,
 3034,
 4244,
 4244,
 3,
 11525,
 63,
 10,
 3,
 23,
 133,
 2816,
 7,
 7,
 1512,
 3,
 99,
 3,
 23,
 1509,
 25,
 11,
 11,
 63,
 95,
 132,
 2629,
 15,
 10,
 147,
 82,
 3654,
 643,
 3,
 55,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0

In [6]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [23]:
train_data[0]["input_ids"]

[3,
 11525,
 63,
 10,
 3,
 32,
 51,
 122,
 410,
 25,
 217,
 3,
 354,
 157,
 48,
 1379,
 58,
 2629,
 15,
 10,
 3,
 23,
 653,
 12,
 1792,
 34,
 16497,
 3,
 11525,
 63,
 10,
 25,
 225,
 43,
 894,
 34,
 34,
 47,
 27635,
 53,
 2629,
 15,
 10,
 3,
 23,
 54,
 17,
 103,
 34,
 7595,
 3,
 23,
 653,
 12,
 3011,
 12,
 8,
 2252,
 16,
 8,
 1379,
 7,
 5,
 5,
 3,
 354,
 157,
 656,
 25,
 317,
 8,
 829,
 296,
 19,
 423,
 13,
 25851,
 7,
 16497,
 3,
 11525,
 63,
 10,
 25,
 164,
 36,
 269,
 3,
 23,
 2483,
 214,
 149,
 128,
 13,
 135,
 54,
 281,
 30,
 132,
 16,
 452,
 21,
 8,
 296,
 12,
 217,
 2629,
 15,
 10,
 3,
 23,
 133,
 67,
 3,
 99,
 3,
 23,
 530,
 3,
 9,
 580,
 12,
 281,
 30,
 132,
 16497,
 2629,
 15,
 10,
 228,
 25,
 3034,
 4244,
 4244,
 3,
 11525,
 63,
 10,
 3,
 23,
 133,
 2816,
 7,
 7,
 1512,
 3,
 99,
 3,
 23,
 1509,
 25,
 11,
 11,
 63,
 95,
 132,
 2629,
 15,
 10,
 147,
 82,
 3654,
 643,
 3,
 55,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0

In [24]:
import torch
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ",device)
model.to(device)


device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500,
)


In [26]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_data,
    eval_dataset = val_data
)

In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.675439,0.382551
2,0.408568,0.357879
3,0.385453,0.352248
4,0.372074,0.349335
5,0.365048,0.346861
6,0.361033,0.346879


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.927935801188151, metrics={'train_runtime': 1266.9709, 'train_samples_per_second': 18.943, 'train_steps_per_second': 2.368, 'total_flos': 3248203235328000.0, 'train_loss': 0.927935801188151, 'epoch': 6.0})

In [29]:
model.save_pretrained("/saved_summery_model ")
tokenizer.save_pretrained("/saved_summery_model ")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/saved_summery_model /tokenizer_config.json',
 '/saved_summery_model /tokenizer.json')

In [ ]:

model = T5ForConditionalGeneration.from_pretrained("/saved_summery_model ")
tokenizer = T5Tokenizer.from_pretrained("/saved_summery_model ")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
model 

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
def summerize_sample(dialogue):
  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
   
      return_tensors="pt"
  )

  targets = model.generate(
      innput_ids = input["input_ids"],
      attention_mask = input["attention_mask"],
      max_length = 150,
      num_beams = 4,
      length_penalty = 0.3,
      early_stopping = True
  )

  summery = tokenizer.decode(targets[0],skip_special_tokens = True )
  return summery

In [31]:
test_dialogue = """Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye"""

def summarize_sample_fixed(dialogue):

    inputs = tokenizer(
        dialogue,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(model.device)

    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        length_penalty=0.3,
        early_stopping=True
    )

    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

# Run the fixed function
summary_result = summarize_sample_fixed(test_dialogue)
print("summary:", summary_result)

summary: he called her last time we were at the park together.
